# Paper Results Defense Notes

This notebook is intentionally separate from the executed RQ1-RQ6 notebooks. It does not retrain models, recompute CAMs, overwrite metric files, or modify saved results. It is a defense and interpretation notebook built from the existing CSV outputs in `outputs/metrics/`.

Use this for paper writing, presentation preparation, and professor Q&A.

## Core Thesis

The main contribution is not simply that Grad-CAM can be displayed on skin lesion images. The stronger and more defensible contribution is a systematic robustness evaluation of CAM explanations under method choice, model architecture, training progression, and external dataset shift.

**Defensible paper claim:** internal HAM10000 performance alone is insufficient. The model can remain highly confident while external discrimination, class-specific sensitivity, calibration, and explanation reliability degrade.

**Avoid overclaiming:** do not say the model is clinically ready. Say that the results show why external validation and explanation robustness checks are necessary before clinical deployment.

In [ ]:
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
METRICS_DIR = NOTEBOOK_DIR / "outputs" / "metrics"

def load_csv(name):
    path = METRICS_DIR / name
    if not path.exists():
        print(f"Missing: {name}")
        return None
    df = pd.read_csv(path)
    print(f"Loaded {name}: {len(df)} rows")
    return df

rq1 = load_csv("RQ1_cam_comparison.csv")
rq2 = load_csv("RQ2_faithfulness.csv")
rq3 = load_csv("RQ3_backbone_comparison.csv")
rq4 = load_csv("RQ4_agreement_results.csv")
rq6_perf = load_csv("RQ6_separate_performance.csv")
rq6_xai = load_csv("RQ6_xai_summary.csv")
rq6_disc = load_csv("RQ6_disagreement_summary.csv")

## RQ1 - CAM Variant Localization

**Question:** Which CAM variant gives the most spatially focused explanation?

**Result from current run:** GradCAM is strongest by both major localization proxies.

| Method | FAP@0.5 lower is better | Entropy lower is better | Interpretation |
|---|---:|---:|---|
| GradCAM | 0.0442 | 7.0243 | Most focused and least diffuse. |
| EigenCAM | 0.0671 | 9.3640 | More diffuse than GradCAM. |
| GradCAM++ | 0.0777 | 10.1709 | More spread out. |
| LayerCAM | 0.0788 | 10.1857 | Similar to GradCAM++, also diffuse. |

**How to explain it:** FAP@0.5 measures how much of the image receives high activation. A smaller value means the explanation is concentrated in a smaller area. Entropy measures how spread out the attention map is. GradCAM being best on both metrics makes the RQ1 conclusion internally consistent.

**Likely question:** Is focused always better?

**Answer:** No. Focus is not the same as clinical correctness. That is why the paper also evaluates faithfulness, architecture effects, agreement, temporal behavior, and external validation. RQ1 only answers which CAM method is most spatially concentrated.

In [ ]:
if rq1 is not None:
    rq1_table = rq1.groupby("method").agg(
        n=("image_id", "count"),
        fap_05_mean=("fap_05", "mean"),
        entropy_mean=("entropy", "mean"),
        confidence_mean=("confidence", "mean"),
    ).sort_values("fap_05_mean")
    display(rq1_table.round(4))

## RQ2 - Faithfulness

**Question:** Do highlighted regions actually influence model confidence?

**Result from current run:** GradCAM is more faithful than GradCAM++.

| Method | Insertion AUC higher is better | Deletion AUC lower is better | Faithfulness higher is better | Positive rate |
|---|---:|---:|---:|---:|
| GradCAM | 0.2193 | 0.1364 | 0.0828 | 62% |
| GradCAM++ | 0.1851 | 0.1610 | 0.0241 | 40% |

**How to explain it:** Insertion tests whether adding the most important pixels back increases confidence. Deletion tests whether removing important pixels decreases confidence. A larger insertion-minus-deletion gap suggests that the highlighted region is more connected to the model decision.

**Likely question:** Does this prove GradCAM is clinically meaningful?

**Answer:** No. It proves relative model faithfulness, not clinical correctness. It shows GradCAM better reflects the model's own decision process than GradCAM++ in this experiment. Clinical validity would require dermatologist-verified lesion regions and prospective testing.

In [ ]:
if rq2 is not None:
    rq2_table = rq2.groupby("method").agg(
        n=("image_id", "count"),
        insertion_auc_mean=("insertion_auc", "mean"),
        deletion_auc_mean=("deletion_auc", "mean"),
        faithfulness_mean=("faithfulness", "mean"),
        positive_faithfulness_rate=("faithfulness", lambda s: float((s > 0).mean())),
    ).sort_values("faithfulness_mean", ascending=False)
    display(rq2_table.round(4))

## RQ3 - Backbone Architecture and XAI Quality

**Question:** Does the classifier backbone affect explanation quality?

**Result from current run:** yes. Accuracy and explanation focus do not move identically.

**Key examples:**

- `resnet50 + GradCAM` had the most focused explanation: FAP@0.5 = 0.0458.
- `efficientnet_b2` had the highest correct rate in this sample: 0.9000.
- But `efficientnet_b2 + GradCAM` was much more diffuse: FAP@0.5 = 0.1781.

**How to explain it:** a backbone can classify well while producing broader or less concentrated CAMs. Therefore, model selection for medical AI should consider both predictive performance and explanation behavior.

**Likely question:** Why not choose only the highest accuracy model?

**Answer:** For a clinical-facing XAI system, the model must be evaluated not only by accuracy but also by whether its explanations are stable and interpretable. RQ3 shows that predictive performance and explanation compactness can diverge.

In [ ]:
if rq3 is not None:
    rq3_table = rq3.groupby(["model", "cam_method"]).agg(
        n=("image_id", "count"),
        correct_rate=("correct", "mean"),
        confidence_mean=("confidence", "mean"),
        fap_05_mean=("fap_05", "mean"),
        entropy_mean=("entropy", "mean"),
    ).sort_values("fap_05_mean")
    display(rq3_table.round(4))

## RQ4 - Inter-Method Agreement vs Correctness

**Question:** If CAM methods agree with each other, does that mean the prediction is correct?

**Result from current run:** no. Incorrect predictions had higher CAM agreement than correct predictions.

| Prediction group | n | Mean confidence | Avg Jaccard |
|---|---:|---:|---:|
| Incorrect | 25 | 0.7584 | 0.5354 |
| Correct | 125 | 0.9161 | 0.2905 |

**How to explain it:** agreement between explanation methods means the methods highlight similar regions. It does not mean those regions are clinically correct or that the class prediction is correct. Several CAM methods can agree on a misleading shortcut.

**Likely question:** Does this contradict the use of CAM agreement?

**Answer:** It contradicts a simplistic trust interpretation. CAM agreement can be reported as a descriptive robustness signal, but it should not be used alone as a correctness or safety signal.

In [ ]:
if rq4 is not None:
    rq4_table = rq4.groupby("correct").agg(
        n=("image_id", "count"),
        confidence_mean=("confidence", "mean"),
        avg_jaccard_mean=("avg_jaccard", "mean"),
        avg_spearman_mean=("avg_spearman", "mean"),
        min_jaccard_mean=("min_jaccard", "mean"),
    )
    rq4_table.index = rq4_table.index.map({0: "Incorrect", 1: "Correct"})
    display(rq4_table.round(4))

## RQ5 - Temporal XAI During Training

**Question:** does Grad-CAM attention evolve as the model learns?

**Result from current run:** validation AUC improved from 0.8211 to about 0.8602 across checkpoints. FAP eventually decreased from 0.1378 to 0.0735, suggesting attention became more focused later in training.

**Important statistical caution:** the AUC-vs-FAP correlation was negative but not significant: `r = -0.3653`, `p = 0.5454`.

**How to explain it:** RQ5 is useful as a temporal visualization and exploratory analysis. It suggests attention may become more concentrated as training progresses, but the evidence is not strong enough to claim a general law.

**Likely question:** Why include RQ5 if it is not statistically significant?

**Answer:** It adds qualitative and temporal insight into model learning. I frame it as exploratory, not confirmatory. It motivates future work with more checkpoints and more images.

## RQ6 - External Validation and Distribution Shift

**Question:** does the HAM10000-trained model remain reliable on external datasets?

**Result from current run:** external performance dropped substantially, while confidence often remained high.

| Dataset | AUC | Accuracy | Mean confidence | Malignant accuracy |
|---|---:|---:|---:|---:|
| HAM10000 | 0.9479 | 0.8699 | 0.8963 | 0.8815 |
| ISIC2020 | 0.6348 | 0.9126 | 0.8936 | 0.2483 |
| MILK10K | 0.6677 | 0.4202 | 0.8625 | 0.2154 |
| Malignant-Benign | 0.7799 | 0.6489 | 0.8655 | 0.3708 |
| PH2 | 0.7450 | 0.5150 | 0.9266 | 0.2000 |

**How to explain it:** high external accuracy can be misleading when the dataset is class-imbalanced. ISIC2020 has high overall accuracy, but malignant accuracy is only 0.2483. That means the model performs well on the dominant benign class but poorly on malignant cases.

**Main RQ6 defense point:** confidence remains high even when external AUC and malignant accuracy drop. This supports a distribution-shift and calibration-risk framing.

In [ ]:
if rq6_perf is not None:
    display(rq6_perf.round(4))

## RQ6 - Explanation Robustness and Zero-CAM Failures

The zero-CAM rate is high across datasets and must be reported honestly.

| Dataset | Zero-CAM rate |
|---|---:|
| HAM10000 | 94.78% |
| ISIC2020 | 94.32% |
| MILK10K | 93.80% |
| Malignant-Benign | 94.21% |
| PH2 | 93.53% |

**How to explain it:** zero CAMs are explanation failures, not successful low-attention explanations. Reporting them makes the paper more credible because it shows the limits of the method instead of hiding them.

**Likely question:** Does the high zero-CAM rate invalidate the work?

**Answer:** It weakens any claim that Grad-CAM is consistently reliable, but it strengthens the robustness argument. The study identifies a failure mode that would be missed if only attractive heatmap examples were shown.

In [ ]:
if rq6_xai is not None:
    display(rq6_xai.round(4))
if rq6_disc is not None:
    display(rq6_disc.round(4))

## Professor Q&A

**Q: What is the single strongest result?**

A: The strongest result is RQ6: internal HAM10000 performance is strong, but external validation exposes lower AUC, poor malignant accuracy, persistent confidence, and explanation failures. That is the most important real-world safety lesson.

**Q: Why not just report accuracy?**

A: Accuracy hides class imbalance. ISIC2020 has high accuracy but poor malignant accuracy. For medical AI, AUC, class-specific accuracy, confidence, and calibration-style metrics must be reported together.

**Q: Are the explanations clinically validated?**

A: No. They are evaluated as model explanations using focus, entropy, faithfulness, agreement, temporal behavior, and external robustness. Clinical validation would require dermatologist annotations or reader studies.

**Q: Why does GradCAM perform better than GradCAM++ here?**

A: In this dataset/model setting, GradCAM produces tighter and more faithful maps. This is an empirical result, not a universal claim that GradCAM always beats GradCAM++.

**Q: Does higher CAM agreement mean higher trust?**

A: No. RQ4 shows the opposite risk: incorrect predictions can have higher CAM agreement. Agreement is descriptive, not a correctness guarantee.

**Q: Why include RQ5 if the p-value is not significant?**

A: RQ5 is exploratory. It shows how attention changes over checkpoints and motivates future temporal XAI work. It should not be framed as a definitive statistical result.

**Q: What would improve the study next?**

A: More external datasets, dermatologist segmentation masks, calibration methods, confidence thresholding, larger temporal checkpoint analysis, and formal saliency sanity checks.

## Claims vs Limitations

**Claims you can defend:**

- GradCAM is the most focused and most faithful CAM method in these experiments.
- Backbone architecture affects explanation behavior, not only predictive behavior.
- CAM agreement should not be treated as a correctness guarantee.
- External validation exposes overconfidence and class-specific failure that internal testing hides.
- Zero-CAM behavior is an explanation failure mode and should be reported.

**Claims to avoid:**

- The system is clinically ready.
- GradCAM explanations are clinically correct.
- High confidence means reliable diagnosis.
- High CAM agreement means correct prediction.
- RQ5 proves attention always becomes clinically meaningful during training.